In [ ]:
!pip install findspark

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
import warnings
warnings.filterwarnings('ignore')
spark_ui_port = 4040
app_name = "Otus"

from itertools import groupby

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .getOrCreate()
)
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter

In [ ]:
spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import subprocess

def process_all_files():

    # Получаем список всех файлов из HDFS
    result = subprocess.run(
        ["hdfs", "dfs", "-ls", "/user/ubuntu/data/"],
        capture_output=True, text=True
    )

    files = []
    for line in result.stdout.strip().split('\n'):
        if '.txt' in line:
            file_path = line.split()[-1]
            file_name = file_path.split('/')[-1].replace('.txt', '')
            files.append(file_name)

    files = sorted(files)
    print(f"\n{'='*70}")
    print(f"НАЙДЕНО ФАЙЛОВ ДЛЯ ОБРАБОТКИ: {len(files)}")
    print(f"{'='*70}")

    results = {}

    for file_date in files:
        print(f"\n{'='*70}")
        print(f"ОБРАБОТКА ФАЙЛА: {file_date}")
        print(f"{'='*70}")

        try:
            # 1. ЧИТАЕМ ИСХОДНЫЙ ФАЙЛ
            print(f"\n1. ЧТЕНИЕ ИСХОДНОГО ФАЙЛА: data/{file_date}.txt")
            rdd = spark.sparkContext.textFile(f"data/{file_date}.txt")
            first_line = rdd.first()

            columns = ['tranaction_id', 'tx_datetime', 'customer_id', 'terminal_id',
                       'tx_amount', 'tx_time_seconds', 'tx_time_days', 'tx_fraud', 'tx_fraud_scenario']

            if first_line.startswith("#"):
                data = rdd.filter(lambda x: not x.startswith("#")).map(lambda x: x.split(","))
                print(f"  Формат: с заголовком #")
            else:
                data = rdd.map(lambda x: x.split(","))
                print(f"  Формат: без заголовка")

            df_raw = spark.createDataFrame(data, schema=columns)
            raw_count = df_raw.count()
            print(f"  Прочитано записей: {raw_count:,}")

            # 2. ПАРТИЦИОНИРОВАНИЕ
            print(f"\n2. СОХРАНЕНИЕ С ПАРТИЦИОНИРОВАНИЕМ ПО tx_time_days")
            (
                df_raw
                    .write
                    .mode("overwrite")
                    .partitionBy("tx_time_days")
                    .parquet(f"data/{file_date}_partitioned.parquet")
            )
            print(f"  Сохранено с партиционированием в: data/{file_date}_partitioned.parquet")

            # 3. ЧИТАЕМ ПАРТИЦИОНИРОВАННЫЕ ДАННЫЕ
            print(f"\n3. ЧТЕНИЕ ПАРТИЦИОНИРОВАННЫХ ДАННЫХ")
            df = spark.read.parquet(f"data/{file_date}_partitioned.parquet")
            print(f"  Прочитано партиций: {df.rdd.getNumPartitions()}")

            # 4. ОБРАБОТКА ДАННЫХ
            print(f"\n4. ОБРАБОТКА ДАННЫХ")

            # Исправляем время 24:00:00
            df = df.withColumn('tx_datetime', F.regexp_replace('tx_datetime', '24:00:00', '00:00:00'))

            # Преобразуем дату в timestamp
            df = df.withColumn('tx_datetime', F.to_timestamp('tx_datetime', 'yyyy-MM-dd HH:mm:ss'))

            # Пустые строки в terminal_id -> NULL
            df = df.withColumn('terminal_id',
                               F.when(F.col('terminal_id') == '', F.lit(None))
                                .otherwise(F.col('terminal_id')))

            # Преобразуем числовые колонки в int
            int_columns = ['tranaction_id', 'customer_id', 'terminal_id',
                           'tx_time_seconds', 'tx_time_days', 'tx_fraud', 'tx_fraud_scenario']
            for col in int_columns:
                df = df.withColumn(col, F.col(col).cast(IntegerType()))

            # Преобразуем сумму в double
            df = df.withColumn('tx_amount', F.col('tx_amount').cast(DoubleType()))

            # Переименовываем колонку
            df = df.withColumnRenamed('tranaction_id', 'transaction_id')

            # Проверка условия fraud
            violations = df.filter((F.col('tx_fraud') == 0) & (F.col('tx_fraud_scenario') != 0)).count()
            if violations > 0:
                print(f"  ВНИМАНИЕ: Найдено {violations} нарушений (fraud=0 но scenario!=0)")
            else:
                print(f"  Проверка fraud: OK")

            final_count = df.count()
            print(f"  После обработки: {final_count:,} записей")

            # 5. ФИНАЛЬНОЕ СОХРАНЕНИЕ В S3
            print(f"\n5. СОХРАНЕНИЕ В S3")
            bucket_name = "otus-bucket-b1g4ki09n8igs1si54v2"
            output_path = f"s3a://{bucket_name}/{file_date}.parquet"

            (
                df.write
                    .mode("overwrite")
                    .parquet(output_path)
            )

            print(f"  Сохранено в: {output_path}")
            print(f"  Записей: {final_count:,}")

            results[file_date] = {
                'raw_count': raw_count,
                'final_count': final_count,
                'violations': violations
            }

        except Exception as e:
            print(f"\n  ОШИБКА: {e}")
            results[file_date] = {'error': str(e)}

    # ФИНАЛЬНЫЙ ОТЧЕТ
    print(f"\n{'='*70}")
    print("ИТОГОВЫЙ ОТЧЕТ ПО ОБРАБОТКЕ 40 ФАЙЛОВ")
    print(f"{'='*70}")

    total_raw = 0
    total_final = 0
    total_violations = 0

    for f, data in results.items():
        if 'error' in data:
            print(f"  ОШИБКА {f}: {data['error']}")
        else:
            raw = data.get('raw_count', 0)
            final = data.get('final_count', 0)
            viol = data.get('violations', 0)
            total_raw += raw
            total_final += final
            total_violations += viol
            print(f"  OK {f}: сырых={raw:,} -> после обработки={final:,} | нарушений={viol}")

    print(f"\n{'='*70}")
    print(f"ВСЕГО:")
    print(f"  Сырых записей: {total_raw:,}")
    print(f"  После обработки: {total_final:,}")
    print(f"  Нарушений fraud логики: {total_violations}")
    print(f"{'='*70}")

    return results

# ЗАПУСК
all_results = process_all_files()